# Alpha: R-Drop soft hierarchical Qwen3-4B QLoRA classification

Enable Internet and a CUDA GPU. This notebook runs classification only: **no OOF or correction experiment**. Attach a private dataset containing `VGEC 2026_v1.json` and `VGEC_classifier.json`. The original seed-42 split, 480 mixed training rows, 81 original holdout rows, five epochs, shared LR 2e-5, rank 16 / alpha 32 / dropout 0.05, and baseline class weights are retained. NF4 double quantization uses BF16 where supported and FP16 otherwise.

The width-five linear `score` head represents one binary error logit and four conditional error-type logits. Joint probabilities are `[1-q, q*r1, q*r2, q*r3, q*r4]`, with `q=sigmoid(z0)` and `r=softmax(z1..z4)`. R-Drop averages weighted BCE + 2*subtype CE from two independent dropout forwards and adds beta=1 times symmetric joint KL, defined as half the sum of both directed KLs. Every term uses the same class weights and full/partial accumulation-window row denominator. Both passes retain gradients; KL is stable FP32. Clean examples have zero supervised subtype gradient, but KL regularizes joint subtype disagreement. Evaluation and saved inference use one forward with dropout disabled and joint argmax, not a hard gate. Training and saved inference share FP32 composition and CUDA autocast at the resolved model precision; CPU has no autocast.

Historical saved-inference Macro-F1: shared-LR baseline 0.45977767, 5× head LR 0.45260493, minimal pairs 0.40740642. Those recipes are inactive and artifacts are preserved. Baseline Trainer BF16 score was 0.43460223 because historical saved inference lacked autocast; baseline must be reevaluated under matching precision. These holdout-selected scores are not independent-test results. **Completed matched five-epoch NF4/BF16 R-Drop: Macro-F1 0.50845162, accuracy 0.69135802; all 81 labels match the coefficient-two hierarchy control, so no classification gain was observed.** Publish `rquq/alpha` before running; no credentials or private data are published.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import signal
import sys

# Use PyTorch; avoid importing optional TensorFlow/Flax stacks.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")

REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "rquq/alpha"
PROJECT_SUBDIR = "."  # Each worktree branch has its project at repository root.
CLASSIFICATION_CONFIG = "configs/classification.json"
CLASSIFICATION_OVERRIDES = {"run": {"seed": 42}, "data": {"path": None, "augmentation_path": None}}
USE_HF_SECRET = False
CLONE_DIR = Path("/kaggle/working/gecc-two-phase-source")
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
print("Branch:", BRANCH, "| config:", CLASSIFICATION_CONFIG)



In [ ]:
# This cell can be rerun after a kernel restart. Git runs as a subprocess.
# Reuse only a clean checkout of the same repository and branch.
from urllib.parse import urlsplit
if urlsplit(REPO_URL).username is not None:
    raise ValueError("Do not embed GitHub credentials in REPO_URL.")
subprocess.run(["git", "check-ref-format", "--branch", BRANCH], check=True, capture_output=True)
if CLONE_DIR.exists():
    if not (CLONE_DIR / ".git").is_dir():
        raise RuntimeError(f"{CLONE_DIR} exists but is not a Git clone; choose another CLONE_DIR.")
    existing_url = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=CLONE_DIR, text=True).strip()
    if existing_url != REPO_URL:
        raise RuntimeError("CLONE_DIR belongs to a different repository; choose another CLONE_DIR.")
    existing_branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=CLONE_DIR, text=True).strip()
    if existing_branch != BRANCH:
        raise RuntimeError("CLONE_DIR uses a different branch; choose another CLONE_DIR.")
    changes = subprocess.check_output(["git", "status", "--porcelain"], cwd=CLONE_DIR, text=True)
    if changes.strip():
        raise RuntimeError("CLONE_DIR has local changes; preserve them and choose another CLONE_DIR.")

remote = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", "--", REPO_URL,
                         f"refs/heads/{BRANCH}"], capture_output=True, text=True)
if remote.returncode == 2:
    raise RuntimeError(f"Branch {BRANCH!r} does not exist on GitHub. Push it or select a published BRANCH.")
if remote.returncode != 0:
    raise RuntimeError("Cannot read the Git repository. Enable Kaggle Internet and check repository access.")
if CLONE_DIR.exists():
    subprocess.run(["git", "fetch", "origin", f"refs/heads/{BRANCH}"], cwd=CLONE_DIR, check=True)
    subprocess.run(["git", "merge", "--ff-only", "FETCH_HEAD"], cwd=CLONE_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
                    "--", REPO_URL, str(CLONE_DIR)], check=True)
PROJECT_DIR = (CLONE_DIR / PROJECT_SUBDIR).resolve()
if not PROJECT_DIR.is_relative_to(CLONE_DIR.resolve()):
    raise ValueError("PROJECT_SUBDIR must stay inside the cloned repository.")
for filename in ["requirements.txt", "model/classification.py", CLASSIFICATION_CONFIG]:
    if not (PROJECT_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing {filename} in {PROJECT_DIR}; check the selected branch and PROJECT_SUBDIR.")
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH, "project_subdir": PROJECT_SUBDIR,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CLONE_DIR, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))



In [ ]:
# Keep the CUDA-enabled torch provided by Kaggle. A constraint prevents
# pip from replacing it while installing the pinned Hugging Face stack.
from importlib.metadata import version
from packaging.version import Version
import tempfile

installed_torch = version("torch")
if Version(installed_torch).base_version != "2.7.1":
    raise RuntimeError(f"requirements.txt targets the HiGANplus stack with CUDA PyTorch 2.7.1; found {installed_torch}. Use a matching image rather than replacing its CUDA build.")
packages = ["transformers", "peft", "accelerate", "safetensors", "huggingface-hub", "tokenizers", "bitsandbytes"]
imported_before = {name: version(name) for name in packages if name.replace("-", "_") in sys.modules}
with tempfile.TemporaryDirectory() as temporary:
    constraints = Path(temporary) / "constraints.txt"
    constraints.write_text(f"torch=={installed_torch}\n", encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt"),
                    "--constraint", str(constraints)], cwd=PROJECT_DIR, check=True)
changed_imports = [name for name, old in imported_before.items() if version(name) != old]
if changed_imports:
    raise RuntimeError(f"Restart the kernel to reload {changed_imports}, then rerun from the parameter cell; the clean clone is reused.")
# Check the training stack directly; unrelated packages in Kaggle's image
# can have pre-existing conflicts that do not affect this PyTorch baseline.
subprocess.run([sys.executable, "-c", "import torch, transformers, peft, accelerate, bitsandbytes; print('Training imports passed')"], check=True)
print("Pinned dependencies installed; retained torch", installed_torch)



In [ ]:
if USE_HF_SECRET:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

def apply_overrides(target, overrides):
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(target.get(key), dict):
            apply_overrides(target[key], value)
        else:
            target[key] = value

config = json.loads((PROJECT_DIR / CLASSIFICATION_CONFIG).read_text(encoding="utf-8"))
apply_overrides(config, CLASSIFICATION_OVERRIDES)
CONFIG_PATH = CLONE_DIR.parent / f"hierarchical-rdrop-{RUN_ID}.json"
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
OUTPUT_DIR = PROJECT_DIR / "runs" / f"hierarchical-rdrop-alpha-{RUN_ID}"
command = [sys.executable, "-m", "model.classification",
           "--config", str(CONFIG_PATH), "--output-dir", str(OUTPUT_DIR), "--skip-hints"]
# Resolve single-GPU precision before loading model weights.
sys.path.insert(0, str(PROJECT_DIR))
from model.classification import placement_preflight
print(json.dumps(placement_preflight(config["model"], config["run"]["device"]), indent=2))


In [ ]:
# Train only the R-Drop hierarchical classifier and save/reload its selected adapter.
# --skip-hints preserves the ablation: no OOF or correction training.
LOG_PATH = OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}.log"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
with LOG_PATH.open("w", encoding="utf-8") as log_file:
    log_file.write(json.dumps(SOURCE_INFO, ensure_ascii=False) + "\n")
    log_file.flush()
    with subprocess.Popen(command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1,
                          start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log_file.write(line)
                log_file.flush()
            returncode = process.wait()
        except BaseException:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


In [ ]:
from IPython.display import JSON, FileLink, display
for filename in ["metrics.json", "run_manifest.json"]:
    print(filename)
    display(JSON(json.loads((OUTPUT_DIR / filename).read_text(encoding="utf-8"))))
if LOG_PATH.is_file():
    display(FileLink(str(LOG_PATH)))
print("R-Drop hierarchical Qwen3 QLoRA classifier:", OUTPUT_DIR / "classifier")
print("Save a notebook version to retain private run outputs.")


## Saved model use

The run retains its resolved config, train/validation IDs, predictions, metrics, epoch history and manifest. The classifier saves the tokenizer, `preprocessing.json`, `classifier_config.json` (exact base revision, actual quantization/dtype, unchanged `soft_hierarchical_error_type` architecture version 2 plus separate `training_algorithm` R-Drop/dropout provenance), and the LoRA adapter with the width-five score head. Existing lambda-two hierarchy adapters remain reloadable; old flat-head checkpoints require their original branch. Checkpoints are selected on the holdout, not an independent test set.

```python
from model.classification import classify
predictions = classify(OUTPUT_DIR / "classifier", records, device="cuda", batch_size=1)
# Each row keeps {id, label, probabilities}; five normalized joint probabilities.
```

For the matched local experiment, run from the alpha worktree:
```bash
python -m model.classification --config ../runs/hierarchical-rdrop-seed42-config.json \
  --output-dir ../runs/hierarchical-rdrop-seed42 --skip-hints
```

The optional connected pipeline remains available through `model.two_phase`, but is outside this experiment. Its reuse path requires a compatible hierarchical checkpoint and complete train-only OOF hints; predicted class 0 still never bypasses correction.
